# Lab | Summarization evaluation using LangSmith
Let's revisit your capstone project 2? Well, sort of. Pick diffierent sets of data and re-run this notebook. Maybe parts of the dataset you used in your last project week. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications using LangSmith.

What did you learn? - Let's discuss that in class

## LangSmith - LangChain evaluation

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
from google.colab import userdata # Import userdata for Colab Secrets

# Load environment variables from a .env file if it exists
_ = load_dotenv(find_dotenv())

# Prioritize loading API keys from Colab Secrets, falling back to os.getenv (for .env or system env vars)
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') or os.getenv('OPENAI_API_KEY')
LANGCHAIN_API_KEY = userdata.get('LANGCHAIN_API_KEY') or os.getenv('LANGCHAIN_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN') or os.getenv('HF_TOKEN')

# Optional: Print status of loaded keys (partially obscured for security)
print(f"OPENAI_API_KEY loaded: {bool(OPENAI_API_KEY)}")
print(f"LANGCHAIN_API_KEY loaded: {bool(LANGCHAIN_API_KEY)}")
print(f"HF_TOKEN loaded: {bool(HF_TOKEN)}")

if not OPENAI_API_KEY:
    print("Warning: OPENAI_API_KEY not found. Please add it to Colab Secrets or a .env file.")
if not LANGCHAIN_API_KEY:
    print("Warning: LANGCHAIN_API_KEY not found. Please add it to Colab Secrets or a .env file.")
if not HF_TOKEN:
    print("Warning: HF_TOKEN not found. Please add it to Colab Secrets or a .env file.")

OPENAI_API_KEY loaded: True
LANGCHAIN_API_KEY loaded: True
HF_TOKEN loaded: True


In [2]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]="langsmith_max-test"

In [3]:
#Importing Client from Langsmith
from langsmith import Client
client = Client(api_key=LANGCHAIN_API_KEY)

### Create Dataset


In [4]:
from datasets import load_dataset
cnn_dataset = load_dataset(
    "cnn_dailymail",
    name="3.0.0"
)

In [5]:
def add_prefix(example):
    return {
        **example,
        "article": f"Summarize this news:\n{example['article']}"
    }

#cnn_dataset = cnn_dataset.map(add_prefix)

In [6]:
cnn_dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [7]:
cnn_dataset['train'][0]

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [8]:
#Get just a few news to test
MAX_NEWS=10
sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS)).map(add_prefix)

sample_cnn

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 10
})

The dataset contains three columns: article, highlights, and id. To use LangSmith, we need to create a dataset in LangSmith format.

LangSmith expects a prompt and a result. To achieve this, we will transform the article into a prompt by adding the prefix: "Summarize this news." As a result, we will use the content of highlights, which represents the summaries created by humans.

In [9]:
print(sample_cnn[0])

{'article': 'Summarize this news:\n(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC\'s founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians\' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki,

Now We have the Dataset with the prompt and the Reference Summary, it is time to create a Dataset in LangSmith with this information.
### Create the Dataset in Langsmith

The dataset in LangSmith is composed of an input, which is the prompt passed to the model for evaluation, and an output, which should contain what we expect the model to return.

In [10]:
import datetime

In [11]:
import uuid
input_key=['article']
output_key=['highlights']

NAME_DATASET=f"Summarize_dataset_{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

In [12]:
#This creates the dataset in LangSmith with the content in sample_cnn - If you run this more than once you will get POST errors
dataset = client.upload_dataframe(
    df=sample_cnn,
    input_keys=input_key,
    output_keys=output_key,
    name=NAME_DATASET,
    description="Test Embedding distance between model summarizations",
    data_type="kv"
)

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

In [13]:
'''
# Replace 'YOUR_LANGCHAIN_API_KEY_HERE' with your actual LangSmith API Key
LANGCHAIN_API_KEY = "YOUR_LANGCHAIN_API_KEY_HERE"

# Set the environment variable for LangChain
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY

# Re-initialize the LangSmith client with the updated API key
from langsmith import Client
client = Client(api_key=LANGCHAIN_API_KEY)

print("LANGCHAIN_API_KEY has been updated and LangSmith client re-initialized.")
print("Please replace 'YOUR_LANGCHAIN_API_KEY_HERE' with your actual key and run this cell.")
'''

'\n# Replace \'YOUR_LANGCHAIN_API_KEY_HERE\' with your actual LangSmith API Key\nLANGCHAIN_API_KEY = "YOUR_LANGCHAIN_API_KEY_HERE"\n\n# Set the environment variable for LangChain\nos.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY\n\n# Re-initialize the LangSmith client with the updated API key\nfrom langsmith import Client\nclient = Client(api_key=LANGCHAIN_API_KEY)\n\nprint("LANGCHAIN_API_KEY has been updated and LangSmith client re-initialized.")\nprint("Please replace \'YOUR_LANGCHAIN_API_KEY_HERE\' with your actual key and run this cell.")\n'

In this image, we can see an example from the dataset once it's been registered in LangSmith.

In the Input column, there is the prompt to be sent, while in the Output column, the expected output is stored.

When performing the comparison, the model will be given the prompt, and the Cosine distance between its response and the one stored in the sample dataset will be calculated.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Dataset.jpg?raw=true">

### Recovering Models From Hugging Face
Let's retrieve both models from HuggingFace. A base T5 model and a model that has been fine-tuned using the training portion of this same dataset to generate summaries.

In [30]:
%pip install -qU langchain-huggingface
from langchain_huggingface import HuggingFaceEndpoint

In [25]:
!pip install -q langchain-community

In [27]:
summarizer_base = HuggingFaceEndpoint(
    repo_id="t5-base",
    temperature=0.1,
    max_new_tokens=180,
    huggingfacehub_api_token=HF_TOKEN
)

/tmp/ipython-input-1270842831.py:1: LangChainDeprecationWarning: The class `HuggingFaceEndpoint` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEndpoint``.
  summarizer_base = HuggingFaceEndpoint(


In [28]:
summarizer_finetuned = HuggingFaceEndpoint(
    repo_id="flax-community/t5-base-cnn-dm",
    temperature=0.1,
    max_new_tokens=180,
    huggingfacehub_api_token=HF_TOKEN
)

## Defining Evaluator
The first step is to define an evaluator, where we specify the variables we want to evaluate. In our case, I have chosen to measure only the "embedding_distance."

I've left the "string_distance" as a comment in case you want to conduct a test with two evaluations instead of one.


In [18]:
!pip install -q langchain

In [19]:
%pip install -q langchain langsmith rapidfuzz langchain-community

In [21]:
# Ensure packages are installed (including classic for smith utilities)
%pip install -qU "langchain" "langchain-classic" "langsmith" "rapidfuzz" "langchain-community"

import langchain
print("LangChain version:", langchain.__version__)

# Try both possible locations for the smith helpers
try:
    from langchain.smith import run_on_dataset, RunEvalConfig
    print("✅ Imported RunEvalConfig from langchain.smith")
except ImportError:
    try:
        from langchain_classic.smith import run_on_dataset, RunEvalConfig
        print("✅ Imported RunEvalConfig from langchain_classic.smith")
    except ImportError:
        print("\n" + "="*80)
        print("ERROR: Neither 'langchain.smith' nor 'langchain_classic.smith' could be imported.")
        print("Possible fixes:")
        print("  - Make sure this cell ran successfully (no pip errors).")
        print("  - Restart the runtime/kernel and run this cell again.")
        print("  - Check that 'langchain-classic' installed correctly.")
        print("="*80 + "\n")
        raise

# Define the evaluation configuration
evaluation_config = RunEvalConfig(
    evaluators=[
        "embedding_distance",   # uses embeddings-based similarity
        # "string_distance",    # uncomment if you also want string distance (needs rapidfuzz)
    ],
)

print("✅ evaluation_config defined.")


LangChain version: 1.1.0
✅ Imported RunEvalConfig from langchain_classic.smith
✅ evaluation_config defined.


### Running Evaluator
With the same configuration, we can launch two evaluations on the same dataset. One for each of the chosen models.

In [23]:
%pip install -q openai tiktoken
import os
from google.colab import userdata

# Ensure OPENAI_API_KEY is set in environment variables for the evaluator
try:
    if "OPENAI_API_KEY" not in os.environ:
        # Try to get it from the variable if it exists, or fetch from userdata
        api_key = globals().get("OPENAI_API_KEY") or userdata.get("OPENAI_API_KEY")
        if api_key:
            os.environ["OPENAI_API_KEY"] = api_key
        else:
            print("Warning: OPENAI_API_KEY not found in globals or secrets.")
except Exception as e:
    print(f"Note: Could not auto-set OPENAI_API_KEY from secrets: {e}")

project_name = f"T5-BASE {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

base_t5_results = run_on_dataset(
    client=client,
    project_name=project_name,
    dataset_name=NAME_DATASET,
    llm_or_chain_factory=summarizer_base,
    evaluation=evaluation_config,
)

View the evaluation results for project 'T5-BASE 2025-11-25 14:56:35' at:
https://smith.langchain.com/o/1577841d-a4fc-4e1b-b2d8-48132552fabb/datasets/7024f85f-5f17-459c-8f20-974f1cd5176d/compare?selectedSessions=33e42910-bba5-4c62-9700-3fc38ff9ad3b

View all tests for Dataset Summarize_dataset_2025-11-25 14:44:23 at:
https://smith.langchain.com/o/1577841d-a4fc-4e1b-b2d8-48132552fabb/datasets/7024f85f-5f17-459c-8f20-974f1cd5176d
[>                                                 ] 0/10

ERROR:langchain_classic.smith.evaluation.string_run_evaluator:Error evaluating run d07b20a8-1929-4f75-bc39-0f57251d20ef
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 810, in _generate_helper
    self._generate(
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 1500, in _generate
    self._call(prompt, stop=stop, run_manager=run_manager, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/langchain_community/llms/huggingface_hub.py", line 138, in _call
    response = self.client.post(
               ^^^^^^^^^^^^^^^^
AttributeError: 'InferenceClient' object has no attribute 'post'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_classic/smith/evaluation/string_run_evaluator.py", line 149, in map
    output_ = self.serialize_outputs(run.outputs)


[--------->                                        ] 2/10

Error Type: AttributeError, Message: 'InferenceClient' object has no attribute 'post'
Error Type: AttributeError, Message: 'InferenceClient' object has no attribute 'post'
ERROR:langchain_classic.smith.evaluation.string_run_evaluator:Error evaluating run d2678309-c453-4396-ad2c-40ba0c854878
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 810, in _generate_helper
    self._generate(
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 1500, in _generate
    self._call(prompt, stop=stop, run_manager=run_manager, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/langchain_community/llms/huggingface_hub.py", line 138, in _call
    response = self.client.post(
               ^^^^^^^^^^^^^^^^
AttributeError: 'InferenceClient' object has no attribute 'post'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  Fi

[------------------------>                         ] 5/10

ERROR:langchain_classic.smith.evaluation.string_run_evaluator:Error evaluating run 2689f492-01ff-42b7-a41d-b4084e684245
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 810, in _generate_helper
    self._generate(
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 1500, in _generate
    self._call(prompt, stop=stop, run_manager=run_manager, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/langchain_community/llms/huggingface_hub.py", line 138, in _call
    response = self.client.post(
               ^^^^^^^^^^^^^^^^
AttributeError: 'InferenceClient' object has no attribute 'post'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_classic/smith/evaluation/string_run_evaluator.py", line 149, in map
    output_ = self.serialize_outputs(run.outputs)


[---------------------------------->               ] 7/10

Error Type: AttributeError, Message: 'InferenceClient' object has no attribute 'post'
Error Type: AttributeError, Message: 'InferenceClient' object has no attribute 'post'
ERROR:langchain_classic.smith.evaluation.string_run_evaluator:Error evaluating run 2b89c4a3-8cce-49e5-b3e2-9075051ac4cb
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 810, in _generate_helper
    self._generate(
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 1500, in _generate
    self._call(prompt, stop=stop, run_manager=run_manager, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/langchain_community/llms/huggingface_hub.py", line 138, in _call
    response = self.client.post(
               ^^^^^^^^^^^^^^^^
AttributeError: 'InferenceClient' object has no attribute 'post'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  Fi

[-------------------------------------------->     ] 9/10

Error Type: AttributeError, Message: 'InferenceClient' object has no attribute 'post'


[------------------------------------------------->] 10/10


In [29]:
#Ignore the error shown below
project_name = f"T5-FineTuned {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

finetuned_t5_results = run_on_dataset(
    client=client,
    project_name=project_name,
    dataset_name=NAME_DATASET,
    llm_or_chain_factory=summarizer_finetuned,
    evaluation=evaluation_config,
)

View the evaluation results for project 'T5-FineTuned 2025-11-25 14:59:07' at:
https://smith.langchain.com/o/1577841d-a4fc-4e1b-b2d8-48132552fabb/datasets/7024f85f-5f17-459c-8f20-974f1cd5176d/compare?selectedSessions=e40332be-2d35-4900-9b9e-225beb486442

View all tests for Dataset Summarize_dataset_2025-11-25 14:44:23 at:
https://smith.langchain.com/o/1577841d-a4fc-4e1b-b2d8-48132552fabb/datasets/7024f85f-5f17-459c-8f20-974f1cd5176d
[>                                                 ] 0/10

ERROR:langchain_classic.smith.evaluation.string_run_evaluator:Error evaluating run 2cf3ec45-fbff-4e51-93df-3a07595ee56d
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 810, in _generate_helper
    self._generate(
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 1500, in _generate
    self._call(prompt, stop=stop, run_manager=run_manager, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/langchain_community/llms/huggingface_endpoint.py", line 267, in _call
    response = self.client.post(
               ^^^^^^^^^^^^^^^^
AttributeError: 'InferenceClient' object has no attribute 'post'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_classic/smith/evaluation/string_run_evaluator.py", line 149, in map
    output_ = self.serialize_outputs(run.outp

[--------->                                        ] 2/10

ERROR:langchain_classic.smith.evaluation.string_run_evaluator:Error evaluating run a9be3184-39c0-49e4-ace5-aa1ef0321618
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 810, in _generate_helper
    self._generate(
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 1500, in _generate
    self._call(prompt, stop=stop, run_manager=run_manager, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/langchain_community/llms/huggingface_endpoint.py", line 267, in _call
    response = self.client.post(
               ^^^^^^^^^^^^^^^^
AttributeError: 'InferenceClient' object has no attribute 'post'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_classic/smith/evaluation/string_run_evaluator.py", line 149, in map
    output_ = self.serialize_outputs(run.outp

[------------------->                              ] 4/10

Error Type: AttributeError, Message: 'InferenceClient' object has no attribute 'post'
ERROR:langchain_classic.smith.evaluation.string_run_evaluator:Error evaluating run a40090e6-de5e-4e82-a323-14026a96b753
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 810, in _generate_helper
    self._generate(
  File "/usr/local/lib/python3.12/dist-packages/langchain_core/language_models/llms.py", line 1500, in _generate
    self._call(prompt, stop=stop, run_manager=run_manager, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/langchain_community/llms/huggingface_endpoint.py", line 267, in _call
    response = self.client.post(
               ^^^^^^^^^^^^^^^^
AttributeError: 'InferenceClient' object has no attribute 'post'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langchain_classic/smith/evaluation/st

... (3 outputs left out)

Error Type: AttributeError, Message: 'InferenceClient' object has no attribute 'post'


[--------------------------------------->          ] 8/10

Error Type: AttributeError, Message: 'InferenceClient' object has no attribute 'post'
Error Type: AttributeError, Message: 'InferenceClient' object has no attribute 'post'


[------------------------------------------------->] 10/10


<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Tests.jpg?raw=true">

In the image below you can see the comparision between two tests.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareTestst.jpg?raw=true">

Well, since it has been so straightforward, why don't we try to make the comparison with an OpenAI model?

In [ ]:
# Use LangChain's official ChatOpenAI to avoid import issues with langchain_openai/langchain_core
from langchain.chat_models import ChatOpenAI
open_aillm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.0)

In [ ]:
project_name = f"OpenAI {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

finetuned_t5_results = run_on_dataset(
    client=client,
    project_name=project_name,
    dataset_name=NAME_DATASET,
    llm_or_chain_factory=open_aillm,
    evaluation=evaluation_config,
)

<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareOpenAI_HF.jpg?raw=true">

The experiment with the OpenAI model has yielded the best results. But, be aware! As we can see, there is a cost involved since we are using an API, and it needs to be paid for.

Another crucial piece of information is that we can view performance data for the models. This data could also be useful for minimally evaluating our inference server.